#   LABORATORIO Nº 1

## Alumnos:
- Araujo, Facundo
- Damaria, Leonel
- Orozco, Lucas
- Varas, María
- Yohena Tanoni, Brenda



## Importación de Librerías

In [ ]:
%pip install matplotlib
%pip install seaborn




In [ ]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns

# Para importación de archivos y acceso a rutas
import glob
import os

## Creacion de Dataframe Original

In [ ]:
# Creacion de Ruta a la Carpeta del Dataset
path_dataset = "Dataset de ventas/*.csv"

# Listar Archivos
file_list = glob.glob(path_dataset)

# Leer y concatenar los archivos
read_file = [pd.read_csv(f) for f in file_list]
original_df = pd.concat(read_file, ignore_index=True)

original_df.head()





### Aclaración: desde este momento, se va trabajar con copia del dataframe original para proteger la conservación de datos

In [ ]:
df = original_df.copy(deep=True)
df.info

## Limpieza de datos
- Eliminar datos no numéricos.
- Eliminar filas incompletas

Se realizó una evaluación de la naturaleza de los datos cargados en el dataframe

 | **[id](ca://s?q=id)** | **[ID Pedido](ca://s?q=ID_Pedido)** | **[Producto](ca://s?q=Producto)** | **[Cantidad Pedida](ca://s?q=Cantidad_Pedida)** | **[Precio Unitario](ca://s?q=Precio_Unitario)** | **[Fecha Pedido](ca://s?q=Fecha_Pedido)** | **[Dirección](ca://s?q=Direccion)** |
|-----------------------|--------------------------------------|-----------------------------------|-----------------------------------------------|-----------------------------------------------|----------------------------------------------|----------------------------------------------|
| **[num](ca://s?q=num)** | **[num](ca://s?q=num)** | **[string](ca://s?q=string)** | **[num](ca://s?q=num)** | **[num](ca://s?q=num)** | **[num](ca://s?q=num)** | **[string](ca://s?q=string)** |


 Hay 7 columnas, de las cuales:

 - 3 tienen valores tipo string (nombre del producto, dirección, fecha)
 - 4 tiene valores tipo numérico (id, id pedido, cantidad pedida, precio).

 Si hay al realizar la limpieza de datos, eliminamos todos los numéricos, terminaríamos sin datos. Por lo que vamos a eliminar los datos no numericos solo de las tablas numéricas, y luego eliminaremos las filas incompletas del resto del dataframe.


In [ ]:
print(" El dataframe original:") 
df.head(3)
print("La cantidad original de filas en el dataframe:", df.shape[0])
print(df.isnull().sum())
print(" Se eliminan valores no numéricos de las columnas Id de pedido, cantidad pedida, precio unitario")
col_num = ["ID de Pedido", "Cantidad Pedida", "Precio Unitario"]
df[col_num] = df[col_num].apply(pd.to_numeric, errors='coerce')
print("Nulos luego de convertir valores no numericos:")
print(df.isnull().sum())
print("Filas antes de la limpieza:", df.shape[0])
df = df.dropna()
print("Limpiando ...")
print("Filas luego de limpieza:", df.shape[0])

df.head(25)



## Preparación de los datos:
- Ajustar los tipos de datos de cada atributo
- Extraer características importantes para el análisis como meses, horas, ciudades etc.

Vamos a crear una clasificación con variables para mejor uso.
- Variable temporal: para poder ver periodos de alta y bajas ventas.
- Variables producto: para saber cuales son los favoritos de los clientes
- Variables de lugar: para poder ver distribución geográfica de las ventas.

### Ajustar los tipos de datos de cada atributo

Para ajustar datos, comenzamos de izquierda a derecha según las columnas.
1. ID Pedido: se lo convierte a número entero y luego en String. No va a ser necesario realizar operaciones entre los números de ID del pedido.

In [ ]:
df['ID de Pedido'] = df['ID de Pedido'].astype(int).astype(str)
df.head(2)

2. Cantidad Pedida: se ajusta el resultado a valores enteros. Al ser elementos tecnoólgicos, no se venden por fracciones.

In [ ]:
df['Cantidad Pedida'] = df['Cantidad Pedida'].astype(int)
df.head(2)

3. Precio Unitario: nos aseguramos que sea  un valor numérico decimal

In [ ]:
df['Precio Unitario'] = df['Precio Unitario'].astype(float)

4. Fecha de Pedido: se ajusta a valores tipo datetime para poder realizar analisis de fechas y horarios de mayores ventas

In [ ]:
df['Fecha de Pedido'] = pd.to_datetime(df['Fecha de Pedido'])
df.head(2)

### Extraer características importantes para el análisis como meses, horas, ciudades etc.
Se extraen las características más importantes para poder mejorar la interpretación de los resultados. Al separar más claramente los meses, la hora de la compra, y sucursal; podremos obtener mapas de calor de ventas de ubicación y hora de ventas.

In [ ]:
# Extraccion de Mes
df['Mes'] = df['Fecha de Pedido'].dt.month

# Extraccion de hora
df['Hora'] = df['Fecha de Pedido'].dt.hour

# Extraccion de Ciudad
df['Ciudad'] = df['Dirección de Envio'].apply(lambda x: x.split(',')[1].strip())

# Extraccion de Venta
df['Venta'] = df['Cantidad Pedida'] * df['Precio Unitario']


df.head()


## Análisis de Datos

Ejemplo:
- Examinar las horas del día en que las ventas son más frecuentes.
- Determinar el mejor momento para mostrar publicidad.

### Meses con más ventas
Fechas Festivas

In [ ]:

# agrupamos por mes, los sumamos y ordenamos de mayor a menor
meses_mas_ventas = df.groupby('Mes')['Venta'].sum().sort_values(ascending=False)
print("los meses con mas ventas: ", meses_mas_ventas)

### Ciudades con más ventas

Lista de las ciudades del dataframe

In [ ]:
sorted(df['Ciudad'].unique())



In [ ]:
# Agrupamos por Ciudad, sumamos las ventas y ordenamos de mayor a menor
ciudades_con_mas_ventas = df.groupby('Ciudad')['Venta'].sum().sort_values(ascending=False)

# Mostrar todas las ciudades ordenadas
print(ciudades_con_mas_ventas)


print("\nLa ciudad con más ventas fue:")
print(ciudades_con_mas_ventas.head(1))

#### Ventas Por Ciudades con Más habitantes
Para poder analizar el alcance en cada ciudad, también tenemos que tener en cuenta, no solo la cantidad total de los ingresos de dinero por ciudad, sino también en cual ciudad más porcentaje de la población es un comprador para no perder la fidelización de los clientes, como en cuales hay que aumentar la publicidad para llegar a más personas.

Tomamos de Google la siguiente información de las Ciudades que aparecen en el dataset.

| **[Ciudad](ca://s?q=Ciudad)** | **[Cantidad de Habitantes](ca://s?q=Cantidad_de_Habitantes)** |
|-------------------------------|------------------------------|
| **[Atlanta](ca://s?q=Atlanta)** | **[530000](ca://s?q=530000)** |
| **[Austin](ca://s?q=Austin)** | **[1000000](ca://s?q=1000000)** |
| **[Boston](ca://s?q=Boston)** | **[675000](ca://s?q=675000)** |
| **[Dallas](ca://s?q=Dallas)** | **[1300000](ca://s?q=1300000)** |
| **[Los Angeles](ca://s?q=Los_Angeles)** | **[3980000](ca://s?q=3980000)** |
| **[New York](ca://s?q=New_York)** | **[8260000](ca://s?q=8260000)** |
| **[Portland](ca://s?q=Portland)** | **[652503](ca://s?q=652503)** |
| **[San Francisco](ca://s?q=San_Francisco)** | **[827526](ca://s?q=827526)** |
| **[Seattle](ca://s?q=Seattle)** | **[781000](ca://s?q=781000)** |


Para eso, creamos un pequeño dataset con las ciudades y sus habitantes


In [ ]:


cant_habitantes = {
    'Ciudad': ['Atlanta', 'Austin', 'Boston', 'Dallas', 'Los Angeles', 'New York', 'Portland', 'San Francisco', 'Seattle'],
    'Habitantes': [530000, 1000000, 675000, 1300000, 3980000, 8260000, 652500, 827526, 781000] 
}
cant_habitantes = pd.DataFrame(cant_habitantes)

!!!! Acá hay que terminar la idea: 
- puede ser, incorporar esta información al otro dataset con un merge para poder hacer comparación.
- descartar la idea por irse por las ramas: no tenemos id de comprador, no sabemos si es lo mismo la dirección de envio que de donde compra, si o no la misma persona que compra mucho. Creo que muchas más dudas como llevarlo a cabo. Pero, si vemos el ejemplo de Atlanta, tiene casi la mitad de habitantes que Austin; pero vendió más.

### Ventas por Ciudad por mes

In [ ]:


# Extraer la Ciudad separando la cadena de texto por las comas
df['Ciudad'] = df['Dirección de Envio'].apply(lambda x: str(x).split(',')[1].strip() if pd.notnull(x) else x)

### Ingreso por hora

In [ ]:
# Ingresos por Hora
pedidos_por_hora = df.groupby('Hora')['ID de Pedido'].count()
print(pedidos_por_hora)

# Ingresos Totales por Hora
ingresos_por_hora = df.groupby('Hora')['Ventas'].sum()
print(ingresos_por_hora)

# Productos por hora
productos_por_hora = df.groupby('Hora')['Cantidad Pedida'].sum()
print(productos_por_hora)

### Cantidad Pedida 
- Media
- Desvío Estándar: ver cuantos productos compran 
- Mínimo y Máximo de las compras
Cuartiles (Q1 - Q3)

### Percentiles de Cantidad Pedida para Stock de Seguridad
Demanda diaria
Percentil de demanda diaria (90 y 95)
Esto sirve para calcular stock
Stock de Seguridad
Cálculo del Stock de Seguridad

### Mejores Productos

### Segmentación del Catálogo por Percentiles de Precio
Low-Ticket P<Q1
Mid-Ticket Q1>P>=Q3
Higth-ticket PQ3


### Mejores Productos por Mes

### Cantidad de venta de los 10 mejores productos según su precio
Varia sus ventas si están en oferta?

### Ranking de venta por mes
La demanda estacional se produce cuando las ventas de ciertos productos se concentran fuertemente en épocas específicas del año (debido al clima, festivos o eventos comerciales)

### Ciclo de Vida del Producto

### Precio Unitario
Media
Desvío Estándar
Minimo y maximo
Cuartil, mediana
rango intercuartil

### Cálculo de KPIs

### Robust Scaling vs. Z-Score
El escalado es vital para preparar los datos antes de aplicar modelos
Z-score (Estandarización): usa la media y el desvio
Robust Scaling (Escalado Robusto) = usa media y el IQR



### Normalización por Categoría e Inferencia ANOVA
Para comparar precios de forma justa entre categorías tan dispares como "Cables" y "Laptops", aplicamos Normalización por Categoría (Z-score calculado individualmente dentro de cada grupo)
ANOVA en Precios Brutos: Obtiene un p-valor = $0.0000$ (se rechaza la hipótesis nula; las diferencias de escala entre categorías son gigantescas y reales).
ANOVA en Precios Normalizados por Categoría: Obtiene un p-valor = $1.0000$. Al normalizar, el promedio de cada categoría se centró exactamente en $0$.



### Formulación de Hipótesis y Pruebas Estadísticas


#### relación entre el Precio Unitario y la Cantidad Pedida 
Pearson :
Spearman :

## Analisis Gráfico

####  Prueba de Hipótesis: Mann-Whitney U (Mejora de variables)
Aplicamos la prueba no paramétrica de Mann-Whitney U para comparar el volumen de compra entre productos baratos (Low-Ticket) y caros (High-Ticket):Hipótesis Nula ($H_0$): La distribución de la cantidad pedida es idéntica entre productos de bajo costo y alto costo.Hipótesis Alternativa ($H_1$): La distribución de la cantidad pedida difiere significativamente entre ambos grupos.


### Tipos de Error en la Toma de Decisiones de Negocio
Error Tipo I (Falso Positivo): Rechazar H cuando es verdadera.
Ejemplo de negocio: Concluir que una campaña de marketing con descuentos aumentó significativamente las ventas (rechazar que las ventas se mantuvieron iguales), cuando en realidad el aumento fue fluctuación aleatoria. La empresa gastará presupuesto en una promoción inútil.
Error Tipo II (Falso Negativo): No rechazar H cuando es falsa (baja potencia estadística).Ejemplo de negocio: Concluir que una nueva tanda de productos importados no tiene una tasa de falla mayor que la estándar (no rechazar la igualdad), cuando en realidad sí venían fallados pero el tamaño de muestra fue muy chico para detectarlo [9]. Esto generará costos masivos ocultos por devoluciones y reclamos de garantía postventa


## Media y Mediana por transacción
Puede verse general y por mes

### Ciudades con más ventas

### Meses con más ventas

In [ ]:
plt.figure(figsize=(10,4))
plt.bar(meses_mas_ventas.index, meses_mas_ventas.values, color="purple")
plt.title("Meses con mayores ventas")
plt.xlabel("Mes")
plt.ylabel("Ventas")
plt.xticks(meses_mas_ventas.index)
plt.show()

In [ ]:
df_meses_ventas =  meses_mas_ventas.reset_index()
df_meses_ventas.columns = ["Mes", "Venta"]
plt.figure(figsize=(10,6))
sns.barplot(data=df_meses_ventas, x="Mes", y="Venta", palette="magma")
plt.title("Meses con mayores ventas (ordenados)")
plt.show()